# 🔬 Notebook 3: Collaborative Whiteboard — Deep Dive


## 🛠️ Setup

```bash
cd 06-system-designs/collaborative-whiteboard
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🗺️ What this notebook covers

We take several ideas and implement a tiny, runnable version of each:

1. **🏚️ Naive concurrency** — why a straightforward "last message wins" server loses edits.
2. **🏛️ Last-Writer-Wins CRDT** — how Lamport + actor tie-break gives deterministic convergence.
3. **🔀 Offline merge** — two users edit while partitioned and the state still converges.
4. **📣 Pub/Sub fan-out** — how WebSocket servers share ops across nodes.
5. **📸 Snapshots + op log** — how late joiners catch up fast.
6. **👻 Presence with TTL** — ephemeral cursors, never persisted.

Everything below runs in-process — no external services.


## 1️⃣ 🏚️ The naive server — and how it loses edits

Imagine the simplest possible server: when a client sends an op, the server just overwrites the shape in an in-memory dict, then broadcasts. No clocks, no merging.

This works *if ops arrive in the same order on every node*. The moment there's any reordering (and there always is — Wi-Fi blips, sharded gateways, mobile networks), edits are **lost**.


In [1]:
# Naive: just apply ops in the order they arrive, no clocks.
class NaiveBoard:
    def __init__(self):
        self.shapes = {}
    def apply(self, op):
        if op["kind"] == "delete":
            self.shapes.pop(op["shape_id"], None)
        else:
            self.shapes[op["shape"]["id"]] = op["shape"]

# Two users edit the same shape "concurrently".
# Server A sees alice's ops first; Server B sees bob's first.
ops_A = [
    {"kind":"add",    "shape":{"id":"s1","x":0,"color":"red"}},
    {"kind":"update", "shape":{"id":"s1","x":10,"color":"red"}},   # alice drags
    {"kind":"update", "shape":{"id":"s1","x":10,"color":"blue"}},  # bob recolors
]
ops_B = [
    {"kind":"add",    "shape":{"id":"s1","x":0,"color":"red"}},
    {"kind":"update", "shape":{"id":"s1","x":10,"color":"blue"}},  # bob first here
    {"kind":"update", "shape":{"id":"s1","x":10,"color":"red"}},   # alice arrives later
]

a, b = NaiveBoard(), NaiveBoard()
for o in ops_A: a.apply(o)
for o in ops_B: b.apply(o)
print("Server A sees:", a.shapes["s1"])
print("Server B sees:", b.shapes["s1"])
print("Diverged! Clients on A see blue, clients on B see red.")


Server A sees: {'id': 's1', 'x': 10, 'color': 'blue'}
Server B sees: {'id': 's1', 'x': 10, 'color': 'red'}
Diverged! Clients on A see blue, clients on B see red.


👉 The naive design **cannot converge** under reordering. We need two things:

1. A **logical timestamp** on every op (Lamport).
2. A **merge rule** that's associative and commutative — same ops, any order, same state.

That's exactly what a CRDT gives us.


## 2️⃣ 🏛️ Last-Writer-Wins (LWW) Map CRDT

Model the board as `shape_id → (timestamp, shape_or_tombstone)`. When two ops touch the same shape, the one with the **higher Lamport** wins. Tie → break on `actor` id (any consistent rule works).

Key property: for any set of ops, **any order** of `apply` yields the **same** final state. That's *convergence*.


In [2]:
from dataclasses import dataclass

@dataclass(order=True, frozen=True)
class Ts:
    lamport: int
    actor: str   # tie-breaker; any stable total order works

class LWWBoard:
    """Board as map shape_id -> (Ts, shape | None). None = tombstone."""
    def __init__(self):
        self.state: dict[str, tuple[Ts, dict | None]] = {}

    def apply(self, op: dict):
        ts = Ts(op["lamport"], op["actor"])
        sid = op["shape_id"] if op["kind"] == "delete" else op["shape"]["id"]
        cur = self.state.get(sid)
        if cur is None or ts > cur[0]:
            if op["kind"] == "delete":
                self.state[sid] = (ts, None)
            else:
                self.state[sid] = (ts, op["shape"])

    def shapes(self) -> list[dict]:
        return [s for _, s in self.state.values() if s is not None]


ops = [
    {"kind":"add",    "shape":{"id":"s1","x":0,"color":"red"},    "lamport":1, "actor":"alice"},
    {"kind":"update", "shape":{"id":"s1","x":10,"color":"red"},   "lamport":3, "actor":"alice"},
    {"kind":"update", "shape":{"id":"s1","x":10,"color":"blue"},  "lamport":3, "actor":"bob"},    # tie on lamport, bob > alice
    {"kind":"update", "shape":{"id":"s1","x":20,"color":"blue"},  "lamport":4, "actor":"bob"},
]

import random
random.seed(0)
# Try several permutations — they must all converge to the same state.
last = None
for perm in [ops, list(reversed(ops)), random.sample(ops, len(ops)), random.sample(ops, len(ops))]:
    b = LWWBoard()
    for o in perm: b.apply(o)
    final = b.shapes()
    print("order -> final:", final)
    if last is not None:
        assert final == last, "diverged!"
    last = final
print("\nEvery order converged to the same state.")


order -> final: [{'id': 's1', 'x': 20, 'color': 'blue'}]
order -> final: [{'id': 's1', 'x': 20, 'color': 'blue'}]
order -> final: [{'id': 's1', 'x': 20, 'color': 'blue'}]
order -> final: [{'id': 's1', 'x': 20, 'color': 'blue'}]

Every order converged to the same state.


### Why does this work?

LWW on a single key is a **last-write-wins register**, a classic CRDT. Promoting it to a *map* of such registers (one per `shape_id`) keeps the CRDT property. Deletes are **tombstones** (the `None` value) so a late-arriving `add` doesn't resurrect a deleted shape.

> ⚖️ **CRDT vs OT (Operational Transform)** — Google Docs historically used OT, which *transforms* ops as they pass through a central server. OT needs a single authoritative ordering and is infamously hard to get right for rich types. CRDTs don't need a central serializer; the cost is slightly more metadata per op (timestamps, tombstones). Modern collaborative apps lean CRDT.


## 3️⃣ 🔀 Offline merge — both users edit while partitioned

This is the *real* test. Alice's laptop loses Wi-Fi on the train. She keeps editing. Bob keeps editing online. When Alice reconnects, both op streams are merged. The final board should be what you'd get if their edits had been interleaved all along.


In [3]:
# Shared starting point
common = [
    {"kind":"add", "shape":{"id":"s1","x":0, "color":"red"},   "lamport":1, "actor":"alice"},
    {"kind":"add", "shape":{"id":"s2","x":0, "color":"green"}, "lamport":2, "actor":"bob"},
]

# While partitioned, each side ticks its own Lamport starting from 2 (last observed).
alice_offline = [
    {"kind":"update", "shape":{"id":"s1","x":50,"color":"red"},    "lamport":3, "actor":"alice"},
    {"kind":"delete", "shape_id":"s2",                             "lamport":4, "actor":"alice"},
]
bob_online = [
    {"kind":"update", "shape":{"id":"s1","x":0, "color":"purple"}, "lamport":3, "actor":"bob"},    # conflicts with alice's move
    {"kind":"add",    "shape":{"id":"s3","x":99,"color":"yellow"}, "lamport":4, "actor":"bob"},
]

def run(all_ops):
    b = LWWBoard()
    for o in all_ops: b.apply(o)
    return sorted(b.shapes(), key=lambda s: s["id"])

alice_view = run(common + alice_offline + bob_online)
bob_view   = run(common + bob_online + alice_offline)   # different interleaving

print("alice final:", alice_view)
print("bob final:  ", bob_view)
assert alice_view == bob_view
print("\nConverged after offline merge. s1 wins by (lamport, actor), s2 stays deleted, s3 added.")


alice final: [{'id': 's1', 'x': 0, 'color': 'purple'}, {'id': 's3', 'x': 99, 'color': 'yellow'}]
bob final:   [{'id': 's1', 'x': 0, 'color': 'purple'}, {'id': 's3', 'x': 99, 'color': 'yellow'}]

Converged after offline merge. s1 wins by (lamport, actor), s2 stays deleted, s3 added.


## 4️⃣ 📣 Pub/Sub fan-out across WebSocket nodes

In production, WebSocket gateways are stateless and many boards span multiple nodes. Pub/sub (Redis, NATS, Kafka…) is how nodes share ops.

Below is a toy in-process bus so you can see the shape of the code.


In [4]:
from collections import defaultdict

class Bus:
    """Tiny synchronous pub/sub. In prod: Redis PUBSUB, NATS, or Kafka."""
    def __init__(self):
        self._subs = defaultdict(list)
    def subscribe(self, topic, fn):
        self._subs[topic].append(fn)
    def publish(self, topic, msg):
        for fn in self._subs[topic]:
            fn(msg)


class WSGateway:
    """Simulates one WS gateway node holding some client connections."""
    def __init__(self, name, bus):
        self.name = name
        self.clients = defaultdict(list)  # board_id -> list[client_name]
        bus.subscribe("board:b1", self._on_msg)

    def attach(self, board_id, client_name):
        self.clients[board_id].append(client_name)

    def _on_msg(self, op):
        # Fan out to our local sockets only — no wasted work.
        for c in self.clients.get(op["board_id"], []):
            payload = op.get("shape", op.get("shape_id"))
            print(f"  [{self.name}] -> {c}: {op['kind']} {payload}")


bus = Bus()
node_A = WSGateway("node-A", bus); node_A.attach("b1", "alice")
node_B = WSGateway("node-B", bus); node_B.attach("b1", "bob"); node_B.attach("b1", "carol")

# Alice sends an op to node-A. node-A validates, assigns lamport, and publishes.
alice_op = {"board_id":"b1","actor":"alice","lamport":5,"kind":"add","shape":{"id":"s9","x":0}}
print("Alice edits. Broadcasting via pub/sub:")
bus.publish("board:b1", alice_op)


Alice edits. Broadcasting via pub/sub:
  [node-A] -> alice: add {'id': 's9', 'x': 0}
  [node-B] -> bob: add {'id': 's9', 'x': 0}
  [node-B] -> carol: add {'id': 's9', 'x': 0}


### Gotchas with pub/sub
- **Echo suppression** — Alice shouldn't re-render her own op. Filter by `actor` on the client.
- **At-most-once vs at-least-once** — Redis Pub/Sub can drop messages during failover. Safe pattern: **durable op log + pub/sub as cache**. On reconnect, clients ask the op log for anything past their `since` cursor.
- **Room affinity** — shard by `board_id` (consistent hash) to keep a board's traffic on fewer nodes and reduce fan-out.


## 5️⃣ 📸 Snapshots + op log — fast catch-up for late joiners

Replaying a million ops to join a meeting would be silly. Every N ops, compress the state to a **snapshot**. Late joiners download the snapshot and replay only the tail.


In [5]:
class OpLog:
    def __init__(self):
        self.ops = []  # append-only
    def append(self, op): self.ops.append(op)
    def since(self, lamport): return [o for o in self.ops if o["lamport"] > lamport]

class Snapshotter:
    def __init__(self, every=3):
        self.every = every
        self.snapshot = None
        self.snapshot_lamport = 0

    def maybe_snapshot(self, log: OpLog):
        if len(log.ops) - self.snapshot_lamport >= self.every:
            b = LWWBoard()
            for o in log.ops: b.apply(o)
            self.snapshot = b.shapes()
            self.snapshot_lamport = max(o["lamport"] for o in log.ops)
            print(f"snapshot at lamport={self.snapshot_lamport}, {len(self.snapshot)} shapes")


log = OpLog(); snap = Snapshotter(every=3)
for i in range(1, 8):
    op = {"board_id":"b1","actor":"alice","lamport":i,"kind":"add",
          "shape":{"id":f"s{i}","x":i*10,"color":"black"}}
    log.append(op)
    snap.maybe_snapshot(log)

# A late joiner: download snapshot, then just replay the tail.
print(f"\nLate joiner catches up: snapshot ({len(snap.snapshot)} shapes) + "
      f"{len(log.since(snap.snapshot_lamport))} tail ops")


snapshot at lamport=3, 3 shapes
snapshot at lamport=6, 6 shapes

Late joiner catches up: snapshot (6 shapes) + 1 tail ops


## 6️⃣ 👻 Presence with TTL — ephemeral cursors

Cursors move 60 times per second. Persisting them would drown any database. Instead:

- Presence flows through pub/sub only.
- Each user's presence has a **TTL** (e.g., 10s). If no heartbeat, drop it.
- On disconnect, publish a `presence_gone` so peers can hide the cursor immediately.


In [6]:
import time

class PresenceRoom:
    def __init__(self, ttl_sec=10.0):
        self.ttl = ttl_sec
        self.cursors = {}  # actor -> (x, y, last_seen)

    def update(self, actor, x, y, now=None):
        self.cursors[actor] = (x, y, now or time.time())

    def active(self, now=None):
        now = now or time.time()
        return {a:(x,y) for a,(x,y,t) in self.cursors.items() if now - t <= self.ttl}


room = PresenceRoom(ttl_sec=2.0)
t0 = 1000.0
room.update("alice", 10, 20, now=t0)
room.update("bob",   30, 40, now=t0 + 0.5)
print("t0+1s  active:", room.active(now=t0 + 1.0))   # both
print("t0+3s  active:", room.active(now=t0 + 3.0))   # both expired
room.update("alice", 11, 21, now=t0 + 3.0)           # heartbeat
print("t0+3.1 active:", room.active(now=t0 + 3.1))   # only alice


t0+1s  active: {'alice': (10, 20), 'bob': (30, 40)}
t0+3s  active: {}
t0+3.1 active: {'alice': (11, 21)}


## 🧠 Closing thoughts

- **CRDTs** (here: LWW map) remove the need for a central serializer → offline-friendly, partition-tolerant.
- **Lamport + actor** is the simplest stable total order that gives deterministic convergence.
- **Snapshots** keep the op log from becoming a liability.
- **Pub/Sub** is the *broadcast mechanism*, not the *source of truth*. The durable op log is the source of truth.
- **Presence is ephemeral** — don't persist it; use TTLs.
- **Validate early** (notebook 2's pydantic) — untrusted clients will send garbage.

### Further reading
- Figma — [How Figma's multiplayer technology works](https://www.figma.com/blog/how-figmas-multiplayer-technology-works/)
- Martin Kleppmann — [CRDTs: Making ∞ data types eventual-consistency-safe](https://martin.kleppmann.com/papers/crdt-hotos21.pdf)
- Yjs docs — [docs.yjs.dev](https://docs.yjs.dev/)
- Automerge — [automerge.org](https://automerge.org/)
- Liveblocks engineering blog — presence and CRDT patterns
